# 02 - Feature Engineering and Labeling

This notebook extracts CSP security features and applies the rule-based scoring system.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.csp_features import FEATURE_COLUMNS, extract_csp_features, label_csp
from src.model_utils import balance_classes

DATA_DIR = PROJECT_ROOT / 'data'

In [ ]:
df = pd.read_csv(DATA_DIR / 'csp_dataset.csv')
df['csp_header'] = df['csp_header'].fillna('')
df.head()

In [ ]:
features = df['csp_header'].apply(extract_csp_features).apply(pd.Series)
scores_labels = df['csp_header'].apply(label_csp)

df_features = pd.concat([df, features], axis=1)
df_features['security_score'] = scores_labels.apply(lambda value: value[0])
df_features['label'] = scores_labels.apply(lambda value: value[1])

df_features[['domain', 'security_score', 'label'] + FEATURE_COLUMNS].head()

## Balance Classes

Balancing is useful because real CSP datasets may contain many missing or weak policies. Here we oversample minority classes for training.

In [ ]:
balanced_df = balance_classes(df_features, label_col='label')
balanced_df['label'].value_counts()

In [ ]:
balanced_df.to_csv(DATA_DIR / 'csp_features_labeled.csv', index=False)
balanced_df.head()